# LLM 中间件与编程范式入门

> 从 Skill、MCP、Harness 到 Agent、Vibe Coding —— 一次搞懂大模型时代的"中间商"生态

---

## 目录

1. [全景图：LLM 应用的三层架构](#1-全景图)
2. [Skill（技能系统）—— 提示词工程的高级形态](#2-skill)
3. [MCP（Model Context Protocol）—— AI 的 USB-C](#3-mcp)
4. [Harness（脚手架）—— 让模型变成 Agent 的执行环境](#4-harness)
5. [Agent（智能体）—— 从工具调用到自主决策](#5-agent)
6. [Vibe Coding —— 用自然语言写代码的新范式](#6-vibe-coding)
7. [它们之间的关系](#7-关系图)
8. [动手练习](#8-动手练习)

<a id='1-全景图'></a>
## 1. 全景图：LLM 应用的三层架构

```
┌─────────────────────────────────────────────────┐
│                   用户 (You)                     │
│              "帮我重构这个函数"                    │
└──────────────────────┬──────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────┐
│            Harness (脚手架/运行时)                │
│  ┌─────────┐  ┌─────────┐  ┌─────────────────┐  │
│  │ Skills  │  │   MCP   │  │     Hooks       │  │
│  │提示模板  │  │外部工具  │  │  确定性自动化    │  │
│  └─────────┘  └─────────┘  └─────────────────┘  │
│  + 权限管理  + 上下文压缩  + 会话管理  + 工具调用  │
└──────────────────────┬──────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────┐
│          LLM (大语言模型, e.g. Claude)            │
│              推理 + 决策 + 生成                   │
└─────────────────────────────────────────────────┘
```

### 核心洞察

| 层 | 职责 | 类比 |
|---|---|---|
| **LLM（模型）** | 理解、推理、生成文本 | 大脑 |
| **Harness（脚手架）** | 工具调度、权限、上下文管理 | 神经系统 + 四肢 |
| **Skill / MCP / Hooks** | 具体的能力扩展 | 技能、工具、条件反射 |

In [ ]:
# 用代码可视化这三层的关系
layers = {
    "LLM (模型)": ["理解自然语言", "推理与规划", "生成代码/文本", "决定调用什么工具"],
    "Harness (脚手架)": ["管理对话上下文", "调度工具调用", "权限控制", "上下文压缩", "会话持久化"],
    "扩展层": {
        "Skill": "预定义的提示模板, 如 /commit, /review",
        "MCP": "外部工具协议, 如查数据库、调API",
        "Hooks": "确定性自动化, 如每次编辑后自动格式化",
    }
}

for layer, capabilities in layers.items():
    print(f"\n{'='*50}")
    print(f"  {layer}")
    print(f"{'='*50}")
    if isinstance(cabilities, dict):
        for k, v in capabilities.items():
            print(f"  {k}: {v}")
    else:
        for cap in capabilities:
            print(f"  - {cap}")

<a id='2-skill'></a>
## 2. Skill（技能系统）—— 提示词工程的高级形态

### 什么是 Skill？

Skill 是 **预定义的提示词模板**，封装了特定任务的完整指令。
用户通过 `/skill-name` 触发，Claude Code 会加载并执行对应的指令。

### Skill 的生命周期

```
会话开始 → 扫描所有 Skill 目录 → 只加载 description 到上下文（极低成本）
     ↓
用户输入 /hello 或匹配到自动触发条件
     ↓
加载完整 SKILL.md → 渲染动态内容（执行 !`command`）→ 注入对话
     ↓
Claude 按照指令执行任务
```

### 文件结构

```
~/.claude/skills/hello/       ← 目录名 = 命令名
  └── SKILL.md               ← 必须有这个文件

~/.claude/skills/review/
  ├── SKILL.md               ← 主指令
  ├── checklist.md           ← 辅助文件
  └── examples/
      └── good-review.md     ← 示例
```

### SKILL.md 格式

In [ ]:
# 展示一个完整的 SKILL.md 示例
skill_example = '''\
---
name: commit-helper
description: 分析当前 git 改动并生成规范的 commit message
argument-hint: "[可选的附加说明]"
disable-model-invocation: true   # 防止 Claude 自动触发，必须手动 /commit-helper
---

## 当前改动

!`git diff HEAD --stat`

!`git diff HEAD`

## 要求

基于以上改动:
1. 用中文总结改动内容（2-3 个要点）
2. 生成一条符合 Conventional Commits 规范的英文 commit message
3. 如果用户提供了附加说明 ($ARGUMENTS)，将其纳入考虑
4. 检查是否有敏感信息（密钥、密码）被意外提交
'''

print(skill_example)

In [ ]:
# Skill 的关键概念总结
skill_concepts = {
    "YAML Frontmatter": {
        "description": "控制 Skill 的行为（名称、描述、是否自动触发等）",
        "关键字段": {
            "description": "描述功能，Claude 据此判断是否自动触发",
            "disable-model-invocation": "true = 只能手动触发，防止副作用",
            "allowed-tools": "声明该 Skill 需要的工具，可跳过权限提示",
            "context: fork": "在隔离的子 Agent 中运行",
        }
    },
    "动态注入": {
        "语法": "!`command`",
        "作用": "在 Claude 看到内容前先执行命令，把输出嵌入 prompt",
        "示例": "!`git diff HEAD` → 实时的代码差异",
    },
    "参数替换": {
        "$ARGUMENTS": "用户输入的全部参数",
        "$1, $2": "按位置获取参数",
    }
}

import json
print(json.dumps(skill_concepts, indent=2, ensure_ascii=False))

<a id='3-mcp'></a>
## 3. MCP（Model Context Protocol）—— AI 的 USB-C

### 什么是 MCP？

MCP 是一个**开放标准协议**，让 AI 应用能以统一的方式连接外部工具和数据源。

```
┌──────────────┐     JSON-RPC 2.0      ┌──────────────┐
│  AI Client   │ ◄──────────────────► │  MCP Server  │
│ (Claude Code) │    stdio / HTTP      │  (你的工具)   │
└──────────────┘                       └──────┬───────┘
                                              │
                                     ┌────────┼────────┐
                                     │        │        │
                                  数据库    API    文件系统
```

### MCP Server 能暴露三种能力

| 能力 | 说明 | 类比 |
|---|---|---|
| **Tools** | 可调用的函数（查数据库、调 API） | 方法/函数 |
| **Resources** | 可读取的数据（文件、API 响应） | 属性/字段 |
| **Prompts** | 预定义的提示模板 | 模板方法 |

### 协议流程

```
1. initialize     → 双方交换能力声明
2. tools/list     → 客户端发现可用工具
3. tools/call     → 客户端调用具体工具
4. (循环 2-3)
```

### 用 Python 写一个 MCP Server

In [ ]:
# ====== 一个完整的 Python MCP Server 示例 ======
# 保存为 server.py, 然后用 claude mcp add 注册

mcp_server_code = '''
from mcp.server.fastmcp import FastMCP

# 初始化服务器
mcp = FastMCP("my-learning-tools")

@mcp.tool()
def add(a: int, b: int) -> int:
    """两数相加

    Args:
        a: 第一个数
        b: 第二个数
    """
    return a + b

@mcp.tool()
def word_count(text: str) -> dict:
    """统计文本的字符数、词数、行数

    Args:
        text: 要统计的文本
    """
    lines = text.split("\\n")
    words = text.split()
    return {
        "chars": len(text),
        "words": len(words),
        "lines": len(lines),
    }

# stdio 模式运行 (与 Claude Code 通信)
mcp.run(transport="stdio")
'''

print(mcp_server_code)

In [ ]:
# MCP Server 注册命令
print("""# 安装依赖
pip install "mcp[cli]"

# 注册到 Claude Code (stdio 模式)
claude mcp add --transport stdio my-tools -- python /path/to/server.py

# 注册远程 MCP Server (HTTP 模式)
claude mcp add --transport http my-api https://api.example.com/mcp

# 查看已注册的 MCP Server
claude mcp list

# 在 Claude Code 中查看状态
# 输入 /mcp
""")

In [ ]:
# 理解 JSON-RPC 2.0 —— MCP 的底层协议
import json

# 客户端 → 服务器: 请求
request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": "add",
        "arguments": {"a": 3, "b": 5}
    }
}

# 服务器 → 客户端: 响应
response = {
    "jsonrpc": "2.0",
    "id": 1,
    "result": {
        "content": [{"type": "text", "text": "8"}]
    }
}

print("请求:")
print(json.dumps(request, indent=2, ensure_ascii=False))
print("\n响应:")
print(json.dumps(response, indent=2, ensure_ascii=False))

In [ ]:
# MCP 关键注意事项
mcp_tips = [
    "1. stdio 模式下绝对不能用 print() —— 会污染 JSON-RPC 数据流，用 logging 模块",
    "2. @mcp.tool() 装饰器自动从类型提示和 docstring 生成 JSON Schema，不用手写",
    "3. 工具描述保持在 2KB 以内 —— Claude Code 会截断过长的描述",
    "4. SDK 选择: Python (mcp[cli]) / TypeScript (@modelcontextprotocol/sdk) / Java/Kotlin/C#/Rust",
    "5. 传输方式: stdio (本地进程) / HTTP (远程服务) / SSE (旧版远程)",
]

for tip in mcp_tips:
    print(tip)

<a id='4-harness'></a>
## 4. Harness（脚手架）—— 让模型变成 Agent 的执行环境

### 什么是 Harness？

Harness = Claude Code 本身。它是将 LLM 从"聊天机器人"升级为"编程 Agent"的完整运行时。

| 组件 | 说明 |
|---|---|
| **工具调度** | 模型决定调用什么工具 → Harness 实际执行 |
| **权限管理** | settings.json 中的 allow/deny/ask 规则 |
| **上下文管理** | 自动压缩过长的对话历史 |
| **Hooks** | 在工具调用前后确定性执行的自动化脚本 |
| **会话管理** | 持久化对话、恢复会话 |

### Hooks —— 确定性自动化

**Hooks vs CLAUDE.md 指令的本质区别：**

| | CLAUDE.md 指令 | Hooks |
|---|---|---|
| 执行者 | 模型（可能忘记） | Harness（100% 执行） |
| 可靠性 | 尽力而为 | 确定性保证 |
| 适用场景 | 偏好、风格指南 | 安全策略、强制格式化 |

In [ ]:
# Hooks 配置示例
hooks_config = {
    "hooks": {
        # 每次编辑后自动格式化
        "PostToolUse": [
            {
                "matcher": "Edit|Write",
                "hooks": [
                    {
                        "type": "command",
                        "command": "npx prettier --write $FILE_PATH"
                    }
                ]
            }
        ],
        # 编辑前检查是否为敏感文件
        "PreToolUse": [
            {
                "matcher": "Edit|Write",
                "hooks": [
                    {
                        "type": "command",
                        "command": "python check_sensitive.py"
                        # exit code 2 = 阻止操作
                    }
                ]
            }
        ],
        # Claude 停止时发送通知
        "Stop": [
            {
                "matcher": "",
                "hooks": [
                    {
                        "type": "command",
                        "command": "notify-send 'Claude Code 完成了任务'"
                    }
                ]
            }
        ]
    }
}

import json
print(json.dumps(hooks_config, indent=2, ensure_ascii=False))

In [ ]:
# Settings.json 的权限配置
permissions_example = {
    "permissions": {
        "allow": [
            "Bash(npm run test *)",      # 允许所有 npm test 命令
            "Bash(git status)",           # 允许 git status
            "Read(./src/**)",             # 允许读取 src 目录
        ],
        "deny": [
            "Read(./.env)",               # 禁止读 .env
            "Bash(rm -rf *)",             # 禁止危险命令
        ],
        "ask": [
            "Bash(curl *)",               # curl 需要每次确认
        ]
    }
}

print("权限评估顺序: deny → ask → allow (先匹配先生效)")
print(json.dumps(permissions_example, indent=2, ensure_ascii=False))

<a id='5-agent'></a>
## 5. Agent（智能体）—— 从工具调用到自主决策

### 什么是 Agent？

Agent = LLM + 工具 + 自主循环。它不只是回答问题，而是**规划、执行、观察、调整**的循环。

### 从 Chatbot 到 Agent 的进化

```
Chatbot:  用户 → LLM → 回答 (单轮)

Tool-Using LLM:  用户 → LLM → [调用工具] → 结果 → LLM → 回答 (1轮工具调用)

Agent:  用户 → LLM → [调用工具] → 观察 → LLM → [再调用工具] → 观察 → ... → 完成
                   ↑__________________________↓
                        自主循环 (Agentic Loop)
```

### Agent 的核心循环

```
    ┌──────────┐
    │  接收任务  │
    └────┬─────┘
         ▼
    ┌──────────┐
    │  规划分解  │  ← 把复杂任务拆成小步骤
    └────┬─────┘
         ▼
    ┌──────────┐
    │  选择工具  │  ← 决定用什么工具 (读文件? 搜代码? 执行命令?)
    └────┬─────┘
         ▼
    ┌──────────┐
    │  执行操作  │  ← Harness 实际执行
    └────┬─────┘
         ▼
    ┌──────────┐
    │  观察结果  │  ← 检查输出是否正确
    └────┬─────┘
         ▼
    ┌──────────┐     ┌──────────┐
    │  完成?    │──否→│  调整策略  │──→ 回到"选择工具"
    └────┬─────┘     └──────────┘
         │ 是
         ▼
    ┌──────────┐
    │  返回结果  │
    └──────────┘
```

In [ ]:
# 用 Python 伪代码理解 Agent 循环
def agent_loop(task: str, tools: dict, llm, max_iterations: int = 10):
    """
    简化的 Agent 循环 - 展示核心逻辑
    
    这就是 Claude Code / OpenAI Agents SDK / LangChain Agent 底层在做的事
    """
    messages = [{"role": "user", "content": task}]
    
    for i in range(max_iterations):
        print(f"\n--- 迭代 {i+1} ---")
        
        # 1. LLM 推理: 决定下一步做什么
        response = llm.chat(messages)
        
        # 2. 检查是否完成
        if not response.tool_calls:
            print(f"Agent 完成: {response.content}")
            return response.content
        
        # 3. 执行工具调用
        for tool_call in response.tool_calls:
            tool_name = tool_call.function.name
            tool_args = tool_call.function.arguments
            
            print(f"  调用工具: {tool_name}({tool_args})")
            
            # Harness 层: 权限检查 + 实际执行
            if tool_name in tools:
                result = tools[tool_name](**tool_args)
            else:
                result = f"错误: 未知工具 {tool_name}"
            
            print(f"  结果: {result}")
            
            # 4. 把结果加入上下文, 进入下一轮
            messages.append({"role": "tool", "content": str(result)})
    
    return "达到最大迭代次数, 任务可能未完成"


# 模拟演示
class MockLLM:
    def __init__(self):
        self.calls = [
            # 第1轮: 先读文件
            type('R', (), {'tool_calls': [type('TC', (), {
                'function': type('F', (), {'name': 'read_file', 'arguments': {'path': 'main.py'}})
            })()], 'content': None})(),
            # 第2轮: 发现 bug, 尝试修复
            type('R', (), {'tool_calls': [type('TC', (), {
                'function': type('F', (), {'name': 'edit_file', 'arguments': {'path': 'main.py', 'fix': 'add null check'}})
            })()], 'content': None})(),
            # 第3轮: 运行测试验证
            type('R', (), {'tool_calls': [type('TC', (), {
                'function': type('F', (), {'name': 'run_command', 'arguments': {'cmd': 'pytest'}})
            })()], 'content': None})(),
            # 第4轮: 完成
            type('R', (), {'tool_calls': None, 'content': '已修复 null pointer bug, 测试全部通过'})(),
        ]
        self.idx = 0
    
    def chat(self, messages):
        r = self.calls[self.idx]
        self.idx += 1
        return r

mock_tools = {
    "read_file": lambda path: f"def process(data):\n    return data['key']  # 可能 KeyError",
    "edit_file": lambda path, fix: f"已修改 {path}: {fix}",
    "run_command": lambda cmd: f"{cmd}: 3 passed",
}

print("模拟 Agent 修复 bug 的过程:\n")
result = agent_loop(
    task="修复 main.py 中的 null pointer bug",
    tools=mock_tools,
    llm=MockLLM()
)

In [ ]:
# Agent 的几种常见模式
agent_patterns = {
    "ReAct (Reason+Act)": {
        "思路": "先思考 → 再行动 → 观察结果 → 再思考",
        "代表": "LangChain ReAct Agent, Claude Code",
    },
    "Plan-and-Execute": {
        "思路": "先制定完整计划 → 再逐步执行",
        "代表": "Claude Code 的 Plan 模式, CrewAI",
    },
    "Multi-Agent": {
        "思路": "多个 Agent 分工协作, 每个 Agent 负责不同任务",
        "代表": "AutoGen, Claude Agent SDK, CrewAI",
    },
    "Tool-Calling": {
        "思路": "模型直接调用工具, 不需要复杂的推理链",
        "代表": "OpenAI Function Calling, Anthropic Tool Use",
    },
}

for name, info in agent_patterns.items():
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    for k, v in info.items():
        print(f"  {k}: {v}")

<a id='6-vibe-coding'></a>
## 6. Vibe Coding —— 用自然语言写代码的新范式

### 什么是 Vibe Coding？

> "You just vibe it into existence." — Andrej Karpathy (2025.2)

Vibe Coding 是一种编程风格：
- 用**自然语言**描述你想要什么
- 让 AI **生成代码**
- 你专注于**意图和方向**，而非语法细节
- 你**审查和验证**产出，而非逐行手写

### 传统编程 vs Vibe Coding

| 维度 | 传统编程 | Vibe Coding |
|---|---|---|
| 输入方式 | 键盘敲代码 | 自然语言描述 |
| 程序员角色 | 实现者 | 导演/审查者 |
| 核心技能 | 语法、算法、调试 | 提示工程、架构判断、测试 |
| 迭代速度 | 分钟级 | 秒级 |
| 风险 | 自己写的 bug 自己知道 | AI 生成的代码需要仔细审查 |

### Vibe Coding 的正确姿势

In [ ]:
# Vibe Coding 的工作流程
vibe_workflow = [
    "1. 描述意图:  '我想做一个网页, 左边是文件树, 右边是代码编辑器'",
    "2. AI 生成初版:  AI 输出完整的 HTML/CSS/JS",
    "3. 试运行 + 调整:  '把左边面板宽度改成 250px, 代码区域用等宽字体'",
    "4. 审查代码:  检查安全隐患、性能问题、边界情况",
    "5. 写测试:  '给这个编辑器组件写单元测试'",
    "6. 迭代优化:  重复 2-5 直到满意",
]

for step in vibe_workflow:
    print(step)

print("\n" + "="*50)
print("Vibe Coding 的关键原则:")
print("="*50)

principles = {
    "验证 > 信任": "AI 生成的代码必须测试, 不能盲目信任",
    "架构 > 细节": "你来决定架构, AI 填充细节",
    "增量 > 全量": "逐步构建, 每步验证, 不要一次生成整个系统",
    "理解 > 复制": "确保你理解 AI 生成的每一行代码的作用",
    "安全 > 方便": "永远检查 SQL 注入、XSS 等安全问题",
}

for principle, desc in principles.items():
    print(f"  {principle}: {desc}")

In [ ]:
# Vibe Coding 的实际例子 - 和 Claude Code 的交互
vibe_examples = [
    {
        "场景": "创建一个 Python CLI 工具",
        "传统方式": "查 argparse 文档 → 写模板 → 加参数 → 调试",
        "Vibe 方式": "'创建一个 CLI 工具, 接受一个 CSV 文件路径, 输出前5行的摘要统计' → 审查生成结果 → 测试",
        "时间对比": "30min vs 3min"
    },
    {
        "场景": "修一个复杂 bug",
        "传统方式": "读代码 → 加 log → 跑 → 读 log → 定位 → 修复",
        "Vibe 方式": "粘贴错误信息 + 描述现象 → AI 分析原因 → 审查修复方案 → 测试",
        "时间对比": "2h vs 15min"
    },
    {
        "场景": "写单元测试",
        "传统方式": "想测试用例 → 写 setUp → 写每个 test → 跑",
        "Vibe 方式": "'给 src/utils.py 写完整的单元测试, 覆盖边界情况' → 审查测试覆盖 → 补充缺失的",
        "时间对比": "1h vs 5min"
    },
]

for ex in vibe_examples:
    print(f"\n{'='*50}")
    print(f"  场景: {ex['场景']}")
    print(f"{'='*50}")
    print(f"  传统: {ex['传统方式']}")
    print(f"  Vibe: {ex['Vibe 方式']}")
    print(f"  时间: {ex['时间对比']}")

<a id='7-关系图'></a>
## 7. 它们之间的关系

### 全景关系图

```
                    ┌─────────────────────────────┐
                    │       Vibe Coding            │
                    │   (编程范式/工作方式)          │
                    │   用自然语言驱动开发            │
                    └──────────────┬──────────────┘
                                   │ 实现工具
                                   ▼
                    ┌─────────────────────────────┐
                    │       Agent (智能体)          │
                    │   自主规划 + 执行 + 调整       │
                    └──────────────┬──────────────┘
                                   │ 运行在
                                   ▼
          ┌─────────────────────────────────────────────┐
          │              Harness (脚手架)                 │
          │    Claude Code / Cursor / Windsurf 等        │
          │                                              │
          │  ┌──────────┐  ┌──────────┐  ┌───────────┐  │
          │  │  Skills  │  │   MCP    │  │   Hooks   │  │
          │  │ (内置能力) │  │(外部工具) │  │ (自动化)   │  │
          │  └──────────┘  └──────────┘  └───────────┘  │
          └─────────────────────────────────────────────┘
```

### 一句话总结每个概念

In [ ]:
# 一句话总结
concepts = {
    "Skill": "封装好的提示词模板, 让 AI 快速执行特定任务 (/commit, /review)",
    "MCP": "标准化的工具协议, 让 AI 能调用外部系统 (数据库、API、文件)",
    "Hook": "确定性的自动化脚本, 在特定事件前后强制执行 (格式化、安全检查)",
    "Harness": "完整的运行时环境, 把 LLM 变成能干活的 Agent", 
    "Agent": "LLM + 工具 + 自主循环 = 能独立完成复杂任务的 AI 系统",
    "Vibe Coding": "用自然语言描述意图, 让 AI 写代码, 你负责审查和决策",
}

for name, desc in concepts.items():
    print(f"  {name:15s} → {desc}")

In [ ]:
# 技术栈对应关系
tech_stack = {
    "Skill": {
        "属于": "Claude Code 特有",
        "类似物": "Cursor Rules, Windsurf Memories",
        "写法": "Markdown + YAML frontmatter",
        "难度": "★☆☆☆☆ (最简单)",
    },
    "MCP": {
        "属于": "开放标准 (Anthropic 发起)",
        "类似物": "OpenAI Function Calling, LangChain Tools",
        "写法": "Python/TypeScript + JSON-RPC",
        "难度": "★★☆☆☆",
    },
    "Hook": {
        "属于": "Claude Code 特有",
        "类似物": "Git Hooks, CI/CD webhooks",
        "写法": "JSON 配置 + Shell 脚本",
        "难度": "★★☆☆☆",
    },
    "Agent": {
        "属于": "通用概念",
        "框架": "Claude Agent SDK, LangChain, CrewAI, AutoGen",
        "写法": "Python/TypeScript 定义工具和循环",
        "难度": "★★★☆☆",
    },
    "Vibe Coding": {
        "属于": "工作方式/范式",
        "工具": "Claude Code, Cursor, Copilot, Windsurf",
        "核心技能": "清晰描述意图 + 代码审查 + 测试",
        "难度": "概念简单, 精通需要经验",
    },
}

for tech, info in tech_stack.items():
    print(f"\n{'='*50}")
    print(f"  {tech}")
    print(f"{'='*50}")
    for k, v in info.items():
        print(f"  {k}: {v}")

<a id='8-动手练习'></a>
## 8. 动手练习

### 练习 1: 创建你的第一个 Skill

In [ ]:
# 练习 1: 创建一个翻译 Skill
# 运行以下命令后, 你就可以在 Claude Code 中用 /translate Hello World 了

translate_skill = '''\
---
name: translate
description: 将文本翻译为指定语言
argument-hint: "[文本] [目标语言]"
---

将以下文本翻译为 $ARGUMENTS 中指定的目标语言。
如果用户没有指定目标语言, 默认翻译为中文。
只输出翻译结果, 不要额外解释。
'''

print("SKILL.md 内容:")
print(translate_skill)
print("\n" + "="*50)
print("部署步骤:")
print("1. mkdir -p ~/.claude/skills/translate")
print("2. 把上面内容保存为 ~/.claude/skills/translate/SKILL.md")
print("3. 重启 Claude Code")
print("4. 输入 /translate Hello World 日语")

### 练习 2: 创建你的第一个 MCP Server

In [ ]:
# 练习 2: 创建一个字符串工具 MCP Server

practice_server = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("string-tools")

@mcp.tool()
def reverse_string(text: str) -> str:
    """反转字符串

    Args:
        text: 要反转的文本
    """
    return text[::-1]

@mcp.tool()
def count_chars(text: str) -> dict:
    """统计字符频率

    Args:
        text: 要统计的文本
    """
    from collections import Counter
    return dict(Counter(text))

@mcp.tool()
def slugify(text: str) -> str:
    """将文本转为 URL 友好的 slug 格式

    Args:
        text: 原始文本
    """
    import re
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\\u4e00-\\u9fff]+", "-", text)
    return text.strip("-")

mcp.run(transport="stdio")
'''

print("server.py 内容:")
print(practice_server)
print("\n" + "="*50)
print("部署步骤:")
print("1. mkdir string-tools && cd string-tools")
print("2. pip install 'mcp[cli]'")
print("3. 把上面内容保存为 server.py")
print("4. claude mcp add --transport stdio string-tools -- python server.py")
print("5. 在 Claude Code 中输入 /mcp 查看 status")

### 练习 3: 用 Vibe Coding 构建一个小项目

In [ ]:
# 练习 3: Vibe Coding 实战指南
print("""
=== Vibe Coding 实战: 构建一个 Markdown 笔记管理 CLI ===

第 1 步: 描述意图 (在 Claude Code 中输入)
─────────────────────────────────────────
"""
step1 = """创建一个 Python CLI 工具 mdnotes, 功能:
- init: 初始化笔记目录 (创建 ~/notes/ 和默认的 index.json)
- new <title>: 创建新的 .md 笔记文件, 自动填充日期和标题模板
- list: 列出所有笔记
- search <keyword>: 在所有笔记中搜索关键词
用 argparse 实现, 代码放在 src/mdnotes/ 下, 要有 pyproject.toml"""

print(f'  输入: "{step1}"\n')

print("""第 2 步: 审查生成结果
─────────────────────────────────────────
  - 检查文件结构是否合理
  - 看 pyproject.toml 的依赖是否正确
  - 检查有没有路径注入漏洞 (os.path.join 中的用户输入)

第 3 步: 增量迭代
─────────────────────────────────────────
  - "加上 --tag 参数, 支持给笔记打标签"
  - "给 list 加上 --sort date 和 --sort title 选项"
  - "加上 export 功能, 把笔记导出为 HTML"

第 4 步: 测试
─────────────────────────────────────────
  - "给 mdnotes 写完整的单元测试, 覆盖所有命令"
  - 手动运行 mdnotes init && mdnotes new test && mdnotes list 验证

第 5 步: 收尾
─────────────────────────────────────────
  - 审查最终代码, 确保理解每一行
  - 加 README (也可以 vibe 出来)
  - git init && git add . && git commit
""")

## 延伸学习资源

| 主题 | 资源 |
|---|---|
| **MCP 官方规范** | https://modelcontextprotocol.io |
| **Claude Code 文档** | https://docs.anthropic.com/en/docs/claude-code |
| **Claude Agent SDK** | https://github.com/anthropics/claude-agent-sdk |
| **Vibe Coding (Karpathy 原文)** | https://x.com/karpathy/status/1886192184808216629 |
| **LangChain (Agent 框架)** | https://python.langchain.com |
| **Anthropic Tool Use 文档** | https://docs.anthropic.com/en/docs/build-with-claude/tool-use |

---

> **下一步**: 从练习 1 开始，创建你的第一个 Skill。5 分钟就能跑起来。